# DAG & Lazy Evaluation

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

The core claim under test: transformations (`.filter()`, `.select()`, `.groupBy()`, `.agg()`) are lazy and never submit a job on their own -- not even `.explain(True)`. Only a real action (`.count()` here) does. Every step below checks this directly against `/api/v1/applications/<id>/jobs`, not just by reading the code.

In [ ]:
import json
import urllib.request

import sys
sys.path.insert(0, "/workspace")

from pyspark.sql import Row, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from driver.playbook import checkpoint

spark = SparkSession.builder.appName("dag-lazy-evaluation").getOrCreate()
app_id = spark.sparkContext.applicationId
print("Spark version:", spark.version)
print("Master:", spark.sparkContext.master)
print("Application id:", app_id)


def jobs_snapshot():
    """Current `/api/v1/applications/<id>/jobs` list -- the ground truth for
    'has a job actually run', queried the same way the topic's Reveal
    self-check panel and the Spark UI Jobs tab do."""
    url = f"http://localhost:4040/api/v1/applications/{app_id}/jobs"
    with urllib.request.urlopen(url, timeout=5) as resp:
        return json.loads(resp.read().decode("utf-8"))

## 1. Build a keyed dataset, then chain filter → select → groupBy → agg -- no action

Every call below (`.filter()`, `.select()`, `.groupBy()`, `.agg()`) only records a step into the logical plan. Building `grouped` here does not run a single task.

Note: `createDataFrame()` is given an **explicit schema** rather than left to infer one from the RDD's rows -- schema inference itself samples the data, which silently runs its own job before the chain even starts. An explicit schema keeps this step genuinely job-free.

In [ ]:
NUM_ROWS = 500_000
NUM_KEYS = 200

schema = StructType([
    StructField("key", StringType(), nullable=False),
    StructField("value", DoubleType(), nullable=False),
])

rdd = spark.sparkContext.parallelize(range(NUM_ROWS), 12).map(
    lambda i: Row(key=f"key-{i % NUM_KEYS}", value=float(i % 997))
)
df = spark.createDataFrame(rdd, schema=schema)

grouped = (
    df.filter(F.col("value") > 100)
    .select("key", "value")
    .groupBy("key")
    .agg(F.count("*").alias("cnt"), F.avg("value").alias("avg_value"))
)
print("Built `grouped` -- type:", type(grouped).__name__)

## 2. Confirm no job has run yet

**Hypothesis:** has building `grouped` above submitted any job? Write your answer before running the cell below.

In [ ]:
jobs = jobs_snapshot()
print(f"Jobs so far: {len(jobs)}")
assert len(jobs) == 0, "expected zero jobs -- nothing but transformations has run yet"

## 3. `.explain(True)` -- compiles the plan, still no job

**Hypothesis:** does printing the parsed/analyzed/optimized/physical plans run anything? `.explain(True)` only asks Catalyst to compile the plan -- it is not an action, no matter how much output it prints.

In [ ]:
grouped.explain(True)

jobs = jobs_snapshot()
print(f"Jobs after .explain(True): {len(jobs)}")
assert len(jobs) == 0, "expected zero jobs -- .explain() compiles the plan, it never runs it"

## 4. Trigger the action -- `.count()`

**Hypothesis:** will a job appear now? `.count()` (called on the DataFrame itself, not the `.groupBy()` step) is a genuine action -- Spark now has to actually run the DAG.

In [ ]:
num_distinct_keys = grouped.count()
print(f"grouped.count() = {num_distinct_keys} distinct keys.")

jobs = jobs_snapshot()
print(f"Jobs after .count(): {len(jobs)}")
assert len(jobs) > 0, "expected at least one job -- .count() is a real action"

## 5. Confirm the DAG's stage boundary lines up with the shuffle

Open **`http://localhost:4040`, Jobs tab** and click into the job `.count()` just submitted -- its DAG visualization should show two stages, split exactly where the `groupBy`'s `Exchange` sits in the physical plan you already saw in step 3. Then call `playbook.checkpoint()` below and click **Reveal self-check** on the topic page: the plan panel labels that `Exchange` as the shuffle boundary, and the stage-metrics panel below it is only populated because a real stage now exists -- empty until this cell ran, which is the same "no job yet" evidence you just checked directly against the REST API.

In [ ]:
checkpoint(grouped, topic="dag-lazy-evaluation")
print("Checkpoint written -- click 'Reveal self-check' on the topic page.")